# Case Study 1: Customer Service Chatbot for Indigo Airlines

In this notebook we build a multi-agent customer service chatbot for Indigo Airlines using the OpenAI Agents SDK — introducing each concept step by step before putting the full system together.

| Section | Concept |
|---|---|
| 1 | Basic agent — no tools, plain text output |
| 2 | Custom tools — letting the agent look things up |
| 3 | Handoffs — routing between specialist agents |
| 4 | Context — shared state across agents |

In [ ]:
# !pip install -q openai-agents python-dotenv

In [ ]:
import json
import time

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel

from agents import (
    Agent,
    FileSearchTool,
    HandoffOutputItem,
    ItemHelpers,
    MessageOutputItem,
    ModelSettings,
    Runner,
    RunContextWrapper,
    function_tool,
    handoff,
)

load_dotenv("../../.env")

## Section 1: A Basic Agent

In [ ]:
basic_agent = Agent(
    name="CustomerServiceAgent",
    instructions="You are a friendly Indigo Airlines customer service agent. Answer customer questions clearly and concisely.",
    model="gpt-4o-mini",
)

In [ ]:
result = await Runner.run(basic_agent, "What is your refund policy?")
print(result.final_output)

In [ ]:
result = await Runner.run(basic_agent, "What is the wifi name on the flight?")
print(result.final_output)

## Section 2: Custom Tools

In [ ]:
def faq_lookup(question: str) -> str:
    question_lower = question.lower()

    if any(word in question_lower for word in ["bag", "baggage", "luggage", "carry-on"]):
        return (
            "You are allowed one carry-on bag. "
            "It must be under 7 kgs and fit in the overhead bin (22x14x9 inches)."
        )

    if any(word in question_lower for word in ["seat", "seats", "seating"]):
        return (
            "The plane has 120 seats: 22 business class and 98 economy. "
            "Rows 5-8 are Economy Plus with extra legroom."
        )

    if any(word in question_lower for word in ["wifi", "internet", "wireless"]):
        return "Free wifi is available on all flights. Connect to 'IndigoDomesticWifi'."

    return "I'm sorry, I don't have an answer for that. Please contact support."

In [ ]:
print(faq_lookup("How heavy can my carry-on bag be?"))
# print(faq_lookup("Is there wifi on the flight?"))
# print(faq_lookup("What seats are available?"))
# print(faq_lookup("Can I bring my pet?"))

### Turning it into an Agent Tool

In [ ]:
@function_tool
def faq_lookup_tool(question: str) -> str:
    """Look up answers to frequently asked questions about the airline."""
    question_lower = question.lower()

    if any(word in question_lower for word in ["bag", "baggage", "luggage", "carry-on"]):
        return (
            "You are allowed one carry-on bag. "
            "It must be under 7 kgs and fit in the overhead bin (22x14x9 inches)."
        )

    if any(word in question_lower for word in ["seat", "seats", "seating"]):
        return (
            "The plane has 120 seats: 22 business class and 98 economy. "
            "Rows 5-8 are Economy Plus with extra legroom."
        )

    if any(word in question_lower for word in ["wifi", "internet", "wireless"]):
        return "Free wifi is available on all flights. Connect to 'IndigoDomesticWifi'."

    return "I'm sorry, I don't have an answer for that. Please contact support."

In [ ]:
# The decorator turns our function into a FunctionTool object.
# This is the exact schema sent to the model — it uses it to decide when and how to call the tool.
print("\n Name        :", faq_lookup_tool.name)
print("\n Description :", faq_lookup_tool.description)
print("\n Parameters  :")
print(json.dumps(faq_lookup_tool.params_json_schema, indent=2))

In [ ]:
faq_agent = Agent(
    name="FAQAgent",
    instructions=(
        "You are a helpful Indigo Airlines Customer Service agent. "
        "Always use the faq_lookup_tool to answer questions — never guess from memory."
    ),
    model="gpt-4o-mini",
    tools=[faq_lookup_tool],
)

result = await Runner.run(faq_agent, "How heavy can my carry-on bag be?")
print(result.final_output)

In [ ]:
result = await Runner.run(faq_agent, "What is the wifi name on the flight?")
print(result.final_output)

In [ ]:
result = await Runner.run(faq_agent, "How many seats does the plane have?")
print(result.final_output)

### `FileSearchTool` — Searching a Real Policy Document

In [ ]:
# The PDF lives in the docs folder — real IndiGo domestic FAQ content.
FAQ_PDF_PATH = "../docs/IndiGo_FAQ.pdf"
print("Using:", FAQ_PDF_PATH)

In [ ]:
client = OpenAI()

# 1. Upload the PDF
with open(FAQ_PDF_PATH, "rb") as f:
    uploaded_file = client.files.create(file=f, purpose="assistants")

# 2. Create a vector store and add the file
vector_store = client.vector_stores.create(name="IndiGo FAQ")

client.vector_stores.files.create(
    vector_store_id=vector_store.id,
    file_id=uploaded_file.id,
)

# 3. Wait for indexing to complete
while True:
    files = client.vector_stores.files.list(vector_store_id=vector_store.id).data
    if files and files[0].status == "completed":
        break
    time.sleep(1)

print("Vector store ready:", vector_store.id)

In [ ]:
faq_pdf_agent = Agent(
    name="IndiGo FAQ Agent",
    instructions=(
        "You are an IndiGo Airlines customer service agent. "
        "Always use the file search tool to answer questions — never rely on your own knowledge."
    ),
    model="gpt-4o-mini",
    tools=[FileSearchTool(vector_store_ids=[vector_store.id])],
)

result = await Runner.run(faq_pdf_agent, "What is the free baggage allowance on domestic flights?")
print(result.final_output)


In [ ]:
result = await Runner.run(faq_pdf_agent, "When does web check-in close for domestic flights?")
print(result.final_output)

### Forcing Tool Use with `tool_choice`

You can override this with `ModelSettings(tool_choice=...)`:

| Value | Behaviour |
|---|---|
| `"auto"` | Model decides (default) |
| `"required"` | Model **must** call at least one tool before replying |
| `"none"` | Model may not call any tools |

In [ ]:
strict_faq_agent = Agent(
    name="IndiGo Strict FAQ Agent",
    instructions=(
        "You are an IndiGo Airlines customer service agent. "
        "Only answer from the policy document. If the document does not contain the answer, say so."
    ),
    model="gpt-4o-mini",
    tools=[FileSearchTool(vector_store_ids=[vector_store.id])],
    model_settings=ModelSettings(tool_choice="required"),  # always search before replying
)

In [ ]:
result = await Runner.run(strict_faq_agent, "How much does it cost to change a Flexi Plus fare flight?")
print(result.final_output)

In [ ]:
result = await Runner.run(strict_faq_agent, "What happens if my bag is lost on a domestic flight?")
print(result.final_output)

## Section 3: Multiple Agents + Handoffs

A single agent can handle anything, but a better pattern for customer service is to have **specialist agents** — each focused on one task — with a **triage agent** that routes the customer to the right one.

The `handoff()` function tells an agent it can transfer the conversation to another agent. The model reads the `handoff_description` of each target agent to decide where to route.

You can inspect `result.new_items` to see exactly what happened — messages, tool calls, and handoffs.

In [ ]:
@function_tool
def update_seat(confirmation_number: str, new_seat: str) -> str:
    """Update the seat for a given booking confirmation number."""
    return f"Seat updated to {new_seat} for confirmation {confirmation_number}."


# --- Specialist agents ---

faq_specialist = Agent(
    name="FAQ Specialist",
    handoff_description="Answers general questions about baggage, seats, wifi, and policies.",
    instructions=(
        "You are an IndiGo Airlines customer service agent. "
        "Always use the file search tool to answer questions — never rely on your own knowledge."
    ),
    model="gpt-4o-mini",
    tools=[FileSearchTool(vector_store_ids=[vector_store.id])],
)

seat_specialist = Agent(
    name="Seat Booking Specialist",
    handoff_description="Handles seat changes and booking updates.",
    instructions=(
        "You are a seat booking agent for Indigo Airlines. "
        "Use whatever confirmation number and seat the customer has already provided. "
        "Only ask for information that is missing, then call update_seat."
    ),
    model="gpt-4o-mini",
    tools=[update_seat],
)

# --- Triage agent ---

triage_agent = Agent(
    name="Triage Agent",
    instructions=(
        "You are the first point of contact for customer service for Indigo Airlines. "
        "Route the customer to the right specialist agent based on their request."
    ),
    model="gpt-4o-mini",
    handoffs=[
        handoff(faq_specialist),
        handoff(seat_specialist),
    ],
)

print("Agents ready.")

In [ ]:
# Run a question that should be routed to the FAQ specialist
result = await Runner.run(triage_agent, "How much can my carry-on bag weigh?")

for item in result.new_items:
    if isinstance(item, HandoffOutputItem):
        print(f"[Handoff] {item.source_agent.name} → {item.target_agent.name}")
    elif isinstance(item, MessageOutputItem):
        print(f"[{item.agent.name}] {ItemHelpers.text_message_output(item)}")

In [ ]:
# Run a request that should be routed to the seat booking specialist
result = await Runner.run(triage_agent, "I'd like to change my seat to 12A. My confirmation is ABC123.")

for item in result.new_items:
    if isinstance(item, HandoffOutputItem):
        print(f"[Handoff] {item.source_agent.name} → {item.target_agent.name}")
    elif isinstance(item, MessageOutputItem):
        print(f"[{item.agent.name}] {ItemHelpers.text_message_output(item)}")

## Section 4: Shared Context (State)

So far each run is stateless — agents don't remember anything between calls. For real customer service you often need to carry **state** across agents and turns, like the customer's name or booking details.

The SDK supports this with a **context** object — a plain Pydantic model you create once and pass into `Runner.run()`. Any agent or tool can read and write it via `RunContextWrapper`.

When a tool accepts `context: RunContextWrapper[YourModel]` as its first argument, the SDK injects it automatically — the model never sees it.

In [ ]:
# --- Context model ---

class BookingContext(BaseModel):
    passenger_name: str | None = None
    confirmation_number: str | None = None
    seat_number: str | None = None


# --- Tools ---

@function_tool
def update_seat(
    context: RunContextWrapper[BookingContext],
    confirmation_number: str,
    new_seat: str,
) -> str:
    """Update the seat for a given booking confirmation number."""
    context.context.confirmation_number = confirmation_number
    context.context.seat_number = new_seat
    return f"Seat updated to {new_seat} for confirmation {confirmation_number}."


# --- Agents typed with BookingContext ---

faq_specialist = Agent[BookingContext](
    name="FAQ Specialist",
    handoff_description="Answers general questions about baggage, seats, wifi, and policies.",
    instructions=(
        "You are an IndiGo Airlines customer service agent. "
        "Always use the file search tool to answer questions — never rely on your own knowledge."
    ),
    model="gpt-4o-mini",
    tools=[FileSearchTool(vector_store_ids=[vector_store.id])],
)

seat_specialist = Agent[BookingContext](
    name="Seat Booking Specialist",
    handoff_description="Handles seat changes and booking updates.",
    instructions=(
        "You are a seat booking agent. "
        "Use whatever confirmation number and seat the customer has already provided. "
        "Only ask for information that is missing, then call update_seat."
    ),
    model="gpt-4o-mini",
    tools=[update_seat],
)

triage_agent = Agent[BookingContext](
    name="Triage Agent",
    instructions=(
        "You are the first point of contact for customer service. "
        "Route the customer to the right specialist agent based on their request."
    ),
    model="gpt-4o-mini",
    handoffs=[
        handoff(faq_specialist),
        handoff(seat_specialist),
    ],
)

print("Agents with context ready.")

In [ ]:
# Create a context object for this customer session
ctx = BookingContext(passenger_name="Alice")

print("\n--- Context before run ---")
print(f"Passenger : {ctx.passenger_name}")
print(f"Confirmation: {ctx.confirmation_number}")
print(f"Seat        : {ctx.seat_number}")

In [ ]:
result = await Runner.run(
    triage_agent,
    "Please change my seat to 14C. My confirmation number is XYZ789.",
    context=ctx,
)

In [ ]:
for item in result.new_items:
    if isinstance(item, HandoffOutputItem):
        print(f"[Handoff] {item.source_agent.name} → {item.target_agent.name}")
    elif isinstance(item, MessageOutputItem):
        print(f"[{item.agent.name}] {ItemHelpers.text_message_output(item)}")

In [ ]:
# Inspect the context after the run — seat and confirmation were saved
print("\n--- Context after run ---")
print(f"Passenger : {ctx.passenger_name}")
print(f"Confirmation: {ctx.confirmation_number}")
print(f"Seat        : {ctx.seat_number}")

In [ ]:
# Second example: a new session, route to FAQ specialist, context stays clean
ctx2 = BookingContext(passenger_name="Rahul")

print("\n--- Context before run ---")
print(f"Passenger : {ctx2.passenger_name}")
print(f"Confirmation: {ctx2.confirmation_number}")
print(f"Seat        : {ctx2.seat_number}")

result = await Runner.run(
    triage_agent,
    "I want to add 10 kg of extra baggage to my flight. How do I do that?",
    context=ctx2,
)

for item in result.new_items:
    if isinstance(item, HandoffOutputItem):
        print(f"[Handoff] {item.source_agent.name} → {item.target_agent.name}")
    elif isinstance(item, MessageOutputItem):
        print(f"[{item.agent.name}] {ItemHelpers.text_message_output(item)}")

print("\n--- Context after run ---")
print(f"Passenger : {ctx2.passenger_name}")
print(f"Confirmation: {ctx2.confirmation_number}")
print(f"Seat        : {ctx2.seat_number}")